## minkunet

In [ ]:
"""from projax3d.io import Scene
import os
REMAP_DICT = {
    1: 1,
    2: 2,
    18: 2,
    31: 2,
    3: 3,
    4: 3,
    5: 3,
    6: 6,
    7: 7,
    9: 9,
    41: 9,
    11: 11,
    14: 14,
    15: 15,
    17: 17,
    19: 19,
    21: 21,
    22: 22,
    25: 25,
    26: 26,
    29: 29,
}


train_dir = r"L:\sitn\lidar22s\Test"
train_files = os.listdir(train_dir)
train_files_paths = [os.path.join(train_dir, file) for file in train_files]
for train_file_path in train_files_paths:
    scene_train = Scene.from_las(
        train_file_path,
        output_parquet = f"L:/raphael/projax-3d-models/data/Test_output_tiles/{(train_file_path.split('\\')[-1]).split('.')[0]}",
        name = "test_output",
        crs="EPSG:2056",
        tile_size=500,
        add_lod=True,
        remap_classification=REMAP_DICT,
        show_progress = True,
    )"""

In [1]:
from projax3d.io import Scene
from projax3d.tiling import GridCuboidSampler
from projax3d.interop.torch import build_model
from projax3d.interop.common import get_feature_config
from projax3d.interop.torch import SceneInference
import os

SITN_CLASS_MAP = {
    2: 1,
    3: 2,
    6: 3,
    7: 0,
    9: 0,
    11: 0,
    14: 0,
    15: 0,
    17: 0,
    19: 0,
    21: 4,
    22: 5,
    25: 0,
    26: 6,
    29: 0,
    1: 0,
}

SITN_CLASS_NAMES = [
    "Other", "Ground", "vegetation", "roof", "cars", "facade", "roof structure"
]

SITN_NUM_CLASSES = 7 


DEVICE = "cuda:0"
VOXEL = 0.25
feature_config = get_feature_config("lidar_standard")

model = build_model("minkunet", {
    "num_classes": SITN_NUM_CLASSES,
    "in_channels": feature_config.num_features,
    "grid_size": VOXEL,
    "channels": (128, 256, 512, 1024),
}).to(DEVICE)

In [2]:
parquet_dirs = r"L:\raphael\projax-3d-models\data\Test_output_tiles"

for folder in os.listdir(parquet_dirs):
    scene_test = Scene.from_parquet(os.path.join(parquet_dirs, folder), crs="EPSG:2056")
    scene_test.compute_footprint()
    sampler = GridCuboidSampler(scene_test, step=(25, 25), size=(40, 40), footprint="auto")
    
    inference = SceneInference.from_checkpoint(
        scene=scene_test,
        checkpoint_path=r"L:\raphael\projax-3d-models\minkunet_v3_weights.pt",
        sampler=sampler,
        model=model,
        output_col="minkunet_v3_prediction",
        device="cuda:0",
        interpolation_k=10,        # KNN IDW back to full point resolution
        output_entropy=True,       # per-point confidence
        class_names=SITN_CLASS_NAMES,
    )
    inference.feature_config = feature_config
    inference._source_columns = feature_config.source_columns
    inference.run()

l:\raphael\1_inference_testset\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Inference:  98%|█████████▊| 387/394 [01:36<00:01,  4.01roi/s, tile=1/1]


## for train set

In [ ]:

from projax3d.io import Scene
from projax3d.tiling import GridCuboidSampler
from projax3d.interop.torch import SceneInference
from projax3d.interop.torch import build_model
from projax3d.interop.common import get_feature_config

SITN_CLASS_NAMES = [
    "Other", "Ground", "vegetation", "roof", "cars", "facade", "roof structure"
]

scene_test = Scene.from_parquet(r"L:\raphael\projax-3d-models\data\Validation", crs="EPSG:2056")
scene_test.compute_footprint()
sampler = GridCuboidSampler(scene_test, step=(25, 25), size=(40, 40), footprint="auto")

SITN_CLASS_MAP = {
    2: 1,
    3: 2,
    6: 3,
    7: 0,
    9: 0,
    11: 0,
    14: 0,
    15: 0,
    17: 0,
    19: 0,
    21: 4,
    22: 5,
    25: 0,
    26: 6,
    29: 0,
    1: 0,
}

SITN_NUM_CLASSES = 7 


DEVICE = "cuda:0"
VOXEL = 0.25
feature_config = get_feature_config("lidar_standard")

model = build_model("minkunet", {
    "num_classes": SITN_NUM_CLASSES,     # 7
    "in_channels": feature_config.num_features,   # 7
    "grid_size": VOXEL,                  # match the voxel sample policy
    "channels": (96, 192, 384, 768),      # wider than the card default → better SITN mIoU
}).to(DEVICE)


inference = SceneInference.from_checkpoint(
    scene=scene_test,
    checkpoint_path=r"L:\raphael\projax-3d-models\minkunet_v100_weights_E87.pt",
    sampler=sampler,
    model=model,
    output_col="minkunet_prediction",
    device="cuda:0",
    interpolation_k=10,        # KNN IDW back to full point resolution
    output_entropy=True,       # per-point confidence
    class_names=SITN_CLASS_NAMES,
)
inference.run(progress=True)

l:\raphael\1_inference_testset\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Inference:  92%|█████████▏| 37488/40969 [4:37:28<25:45,  2.25roi/s, tile=93/93]    


93

## ptv3

In [1]:
from projax3d.io import Scene
from projax3d.tiling import GridCuboidSampler
from projax3d.interop.torch import SceneInference
SITN_CLASS_NAMES = [
    "Other", "Ground", "vegetation", "roof", "cars", "facade", "roof structure"
]

scene_test = Scene.from_parquet(r"L:\raphael\projax-3d-models\data\Test_small", crs="EPSG:2056")
scene_test.compute_footprint()
sampler = GridCuboidSampler(scene_test, step=(40, 40), size=(40, 40), footprint="auto")

In [2]:
import torch
from projax3d.io import Scene
from projax3d.tiling import GridCuboidSampler
from projax3d.interop.torch import SceneLoader, build_model, CombinedFocalLovaszLoss
from projax3d.interop.common import get_feature_config
from projax3d.interop.common.feature_config import FeatureConfig, op_scale

SITN_CLASS_MAP = {
    2: 1,
    3: 2,
    6: 3,
    7: 0,
    9: 0,
    11: 0,
    14: 0,
    15: 0,
    17: 0,
    19: 0,
    21: 4,
    22: 5,
    25: 0,
    26: 6,
    29: 0,
    1: 0,
}

SITN_NUM_CLASSES = 7 


DEVICE = "cuda:0"
VOXEL = 0.5
feature_config = get_feature_config("lidar_ptv3")


model = build_model("ptv3_seg", {
    "num_classes": SITN_NUM_CLASSES,
    "in_channels": feature_config.num_features,
    "grid_size": VOXEL,

    "backbone_out_channels": 256,

    "backbone_config": {
        "enc_depths": (2, 3, 6, 8, 3),
        "enc_channels": (96, 192, 384, 768, 768),
        "enc_num_head": (6, 12, 24, 48, 48),

        "dec_depths": (3, 3, 3, 3),
        "dec_channels": (256, 256, 384, 768),
        "dec_num_head": (8, 8, 12, 24),

        "enc_patch_size": (256,256,256,256,256),
        "dec_patch_size": (256,256,256,256),

        "mlp_ratio": 4,
        "drop_path": 0.2,
        "enable_flash": True,
    }
}).to(DEVICE)

l:\raphael\1_inference_testset\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
inference = SceneInference.from_checkpoint(
    scene=scene_test,
    checkpoint_path=r"l:\raphael\projax-3d-models\ptv3_seg_weights_v100.pt",
    sampler=sampler,
    model=model,
    output_col="pvt3_prediction_large",
    device="cuda:0",
    interpolation_k=10,        # KNN IDW back to full point resolution
    output_entropy=True,       # per-point confidence
    class_names=SITN_CLASS_NAMES,
)
inference.run(progress=True)

Inference: 100%|██████████| 169/169 [02:44<00:00,  1.03roi/s, tile=1/1]


1